# 03 · 블록·sub-block 비용 모델

블록 캐시와 DualCache의 trade-off를 임의 단위 비용으로 탐색한다. 하드웨어 benchmark나 논문 가속 수치의 재현이 아니다.

**학습 목표**: block·sub-block 크기와 cache 가정이 toy latency와 처리량 최적점에 미치는 영향을 해석한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 표준 라이브러리 `math`만 사용하며 외부 패키지는 없다.

In [ ]:
# ceil은 마지막 불완전 block도 실제 작업 한 번으로 계산하기 위해 사용한다.
from math import ceil

def toy_latency(length, block, subblock, tokens_per_round, cache=True):
    blocks = ceil(length / block)
    rounds = ceil(block / tokens_per_round)
    # 고정 launch 비용 + 현재 sub-block attention + prefix read의 임의 비용
    launch = blocks * rounds
    local_attention = blocks * rounds * subblock / 16
    prefix_read = (blocks * (blocks - 1) / 2) * (0.10 if cache else block * 0.10)
    mismatch = 0 if block % subblock == 0 else 2.0 * blocks
    return launch + local_attention + prefix_read + mismatch


In [ ]:
rows = []
for block in (16, 32, 64):
    for subblock in (4, 8, 16):
        latency = toy_latency(256, block, subblock, tokens_per_round=4, cache=True)
        rows.append((latency, block, subblock, 256 / latency))

for latency, block, subblock, throughput in sorted(rows):
    print(f'block={block:2d} sub={subblock:2d} latency={latency:6.2f} throughput={throughput:5.2f}')
best = min(rows)
assert best[1] % best[2] == 0
print('toy optimum:', {'block': best[1], 'subblock': best[2]})


실제 최적점은 kernel, batch, sequence length와 승인되는 token 수에 따라 바뀐다. 논문의 block 32/sub-block 8을 보편적 상수로 취급하지 말고 장치별 profile을 수행해야 한다.